# 5074project 1
**Group 6**
Changyang Zhu cz3003
Yizhang Mu ym 3178


**GitHub repository:** https://github.com/deepthink869/qmss5074group6fall2026

##write in the front
This notebook compares Random Forest, SVC and neural networks for classifying 137 countries into five happiness groups.

The two datasets cover different periods: the development indicators are from around 2015–2017, and the happiness surveys cover 2020–2022. We use `happiness_score` to create the labels and then remove it from the predictors.For the neural networks, we split those 95 countries into 71 for fitting and 24 for validation.

To run the notebook, keep the `data/` folder beside it, install `requirements.txt`, and run all cells from a fresh kernel.

load data

In [ ]:
import importlib.util
import subprocess
import sys
packages = {'pandas': 'pandas', 'numpy': 'numpy', 'sklearn': 'scikit-learn',
            'scipy': 'scipy', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn',
            'keras': 'keras', 'torch': 'torch', 'shap': 'shap', 'joblib': 'joblib'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'
os.environ['OMP_NUM_THREADS'] = '1'
import random, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.dpi': 115, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'figure.facecolor': 'white',
                     'axes.prop_cycle': plt.cycler(color=['#2463A0','#D18B20','#7652A1','#278477','#B4516B'])})

print('Python:', platform.python_version(), '| Keras backend: torch (CPU)')
whr_df = pd.read_csv('data/WHR_2023.csv')
assert len(whr_df) == 137 and whr_df['country'].is_unique
display(whr_df.head(2))

In [ ]:
CLASS_NAMES = ['Very Low', 'Low', 'Average', 'High', 'Very High']
whr_df['happiness_category'] = pd.qcut(whr_df['happiness_score'], q=5, labels=CLASS_NAMES)
X = whr_df.drop(columns=['happiness_score', 'happiness_category'])
y = whr_df['happiness_category']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
y_train_labels = y_train.astype('category').cat.codes
y_test_labels = y_test.astype('category').cat.codes
print('Rows: train', len(y_train), '| test', len(y_test))
print('Training class counts:', y_train.value_counts(sort=False).to_dict())
print('Encoded target labels:', dict(enumerate(CLASS_NAMES)))

y_train` holds the category names, while `y_train_labels` holds the corresponding integers. Both keep the same country order.

In [ ]:
countrydata = pd.read_csv('data/newcountryvars.csv')
print('Development dataset shape:', countrydata.shape)
display(countrydata.head(2))

name_map = {
    'Congo': 'Congo (Brazzaville)',
    'Democratic Republic of the Congo': 'Congo (Kinshasa)',
    'Czech Republic': 'Czechia',
    'Hong Kong': 'Hong Kong S.A.R. of China',
    'Macedonia': 'North Macedonia',
    'Palestine': 'State of Palestine',
    'Turkey': 'Turkiye',
    'VietNam': 'Vietnam',
}
countrydata['country_name'] = countrydata['country_name'].replace(name_map)

common_countries = set(X['country']) & set(countrydata['country_name'])
unmatched = sorted(set(X['country']) - common_countries)
print(f'Exact/harmonized country matches: {len(common_countries)}/{len(X)}')
print('Countries without development records:', unmatched)
print('Duplicate country names in development data:', countrydata['country_name'].duplicated().sum())

In [ ]:
train_country_order = X_train['country'].tolist()
test_country_order = X_test['country'].tolist()
X_train = (X_train.merge(countrydata, how='left', left_on='country',
                         right_on='country_name', sort=False, validate='one_to_one')
           .drop(columns='country_name').reset_index(drop=True))
X_test = (X_test.merge(countrydata, how='left', left_on='country',
                       right_on='country_name', sort=False, validate='one_to_one')
          .drop(columns='country_name').reset_index(drop=True))
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert not X_train['country'].duplicated().any()
print('Merged train:', X_train.shape, '| test:', X_test.shape)
print('Country with no matched development data:', unmatched)
assert X_train['country'].tolist() == train_country_order
assert X_test['country'].tolist() == test_country_order

After matching those names, 136 of the 137 WHR countries have development records. We keep Taiwan in the dataset and impute its missing development indicators later, because Taiwan count in China. The left join keeps the countries in the same order as their labels.

## exploratory analysis

In [ ]:
print(X_train.dtypes)

We drop `country` from the model inputs and one-hot encode `region`. The other columns are numeric; we check their missing values and distributions below.

### Missing values

In [ ]:
missing_summary = pd.DataFrame({
    'missing_n': X_train.isna().sum(),
    'missing_pct': (100 * X_train.isna().mean()).round(1),
}).sort_values(['missing_n', 'missing_pct'], ascending=False)
print('Missing values in training data (n=95):')
display(missing_summary)

In [ ]:
num_columns = X_train.select_dtypes(include='number').columns.tolist()
plot_vars = ['gdp_per_capita', 'social_support', 'hdi', 'population', 'gni',
             'population_below_poverty_line']
fig, axs = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axs.flat, plot_vars):
    ax.hist(X_train[col].dropna(), bins=16, edgecolor='white')
    ax.set_title(col.replace('_', ' '), fontsize=10)
    ax.set_ylabel('Training countries')
    ax.set_xlabel({'population': 'People', 'gni': 'PPP international dollars per person',
                   'hdi': 'Index (0–1)', 'population_below_poverty_line': 'Percent',
                   'gdp_per_capita': 'WHR contribution (score points)',
                   'social_support': 'WHR contribution (score points)'}[col])
fig.suptitle('Selected numerical feature distributions (training set only)', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
X_train['region'].value_counts().plot(kind='barh', ax=axs[0], title='Training countries by region')
pd.Series(y_train.to_numpy(), name='happiness_category').value_counts().reindex(CLASS_NAMES).plot(
    kind='bar', ax=axs[1], title='Training countries by happiness category')
axs[1].tick_params(axis='x', rotation=30)
axs[0].set(xlabel='Number of training countries', ylabel='Region')
axs[1].set(xlabel='Happiness category', ylabel='Number of training countries')
plt.tight_layout(); plt.show()
print('Targets are balanced in train (19 per category) by stratified splitting.')
print('Regions are NOT balanced; the smallest have just two countries.')

### correlations

In [ ]:
import seaborn as sns
corr_matrices = {method: X_train[num_columns].corr(method=method)
                 for method in ('pearson', 'spearman', 'kendall')}
short_labels = {'gdp_per_capita':'GDP contribution','social_support':'Social support',
    'healthy_life_expectancy':'Health contribution','freedom_to_make_life_choices':'Freedom',
    'generosity':'Generosity','perceptions_of_corruption':'Corruption contribution',
    'population':'Population','population_below_poverty_line':'Poverty (%)','hdi':'HDI',
    'life_expectancy':'Life expectancy','expected_years_of_schooling':'Expected schooling',
    'mean_years_of_schooling':'Mean schooling','gni':'GNI per capita'}
fig, axes = plt.subplots(1, 3, figsize=(21, 7), layout='constrained')
for ax, (method, matrix) in zip(axes, corr_matrices.items()):
    labelled = matrix.rename(index=short_labels, columns=short_labels)
    sns.heatmap(labelled, vmin=-1, vmax=1, cmap='RdBu_r', center=0,
                square=True, cbar=(method == 'kendall'), ax=ax)
    ax.set_title(f'{method.title()} correlation — training set')
    ax.tick_params(labelsize=9)
plt.show()

HDI is strongly correlated with schooling (Pearson r = 0.938) and life expectancy (r = 0.932). Spearman and Kendall check monotonic associations. These overlapping predictors can share model importance.

### Relationships with happiness

In [ ]:
eda_train = X_train.assign(happiness_category=y_train.to_numpy(),
                           target_rank=y_train_labels.to_numpy())
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
sns.boxplot(data=eda_train, x='happiness_category', y='gdp_per_capita',
            order=CLASS_NAMES, ax=axes[0])
sns.boxplot(data=eda_train, x='happiness_category', y='social_support',
            order=CLASS_NAMES, ax=axes[1])
sns.scatterplot(data=eda_train, x='hdi', y='gni',
                hue='happiness_category', hue_order=CLASS_NAMES, ax=axes[2], s=45)
axes[2].set_yscale('log')
for ax in axes[:2]:
    ax.tick_params(axis='x', rotation=30)
    ax.set_xlabel('Happiness category')
axes[0].set_ylabel('GDP contribution (score points)')
axes[1].set_ylabel('Social support contribution (score points)')
axes[2].set(xlabel='HDI', ylabel='GNI per capita (PPP dollars, log scale)')
plt.tight_layout(); plt.show()
class_means = eda_train.groupby('happiness_category', observed=False)[
    ['gdp_per_capita', 'social_support', 'hdi', 'life_expectancy']].mean().reindex(CLASS_NAMES)
display(class_means.round(3))
association = eda_train[num_columns + ['target_rank']].corr(method='spearman')[
    'target_rank'].drop('target_rank').sort_values(ascending=False)
display(association.head(6).rename('Spearman correlation with class').round(3).to_frame())

In [ ]:
q1, q3 = X_train[num_columns].quantile(.25), X_train[num_columns].quantile(.75)
iqr = q3 - q1
flags = X_train[num_columns].lt(q1 - 1.5 * iqr) | X_train[num_columns].gt(q3 + 1.5 * iqr)
outlier_count = flags.sum().sort_values(ascending=False).rename('IQR flags')
display(outlier_count.to_frame())
print('Most-flagged countries (by number of variables):')
flagged = X_train[['country']].copy()
flagged['flag_count'] = flags.sum(axis=1)
display(flagged.sort_values('flag_count', ascending=False).head(10))

The higher happiness groups tend to have higher GDP contribution, social support, HDI and life expectancy. Each training class has 19 countries, but some regions have only two.

Population and GNI have long right tails. HDI, schooling and life expectancy overlap strongly; a model may use one in place of another.

## feature engineering

In [ ]:
for frame in (X_train, X_test):
    frame['log_population'] = np.log1p(frame['population'])
    frame['log_gni'] = np.log1p(frame['gni'])
    frame['gdp_x_social_support'] = frame['gdp_per_capita'] * frame['social_support']
    frame['hdi_x_schooling'] = frame['hdi'] * frame['mean_years_of_schooling']

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = ['region']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop', sparse_threshold=0)

preprocess = preprocessor.fit(X_train)
print('Preprocessing fitted on', len(X_train), 'training countries only.')
print('Transformed features:', len(preprocess.get_feature_names_out()))

In [ ]:
def preprocessor(data):
    return preprocess.transform(data)

X_train_processed = preprocessor(X_train)
X_test_processed = preprocessor(X_test)
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert np.isfinite(X_train_processed).all() and np.isfinite(X_test_processed).all()
print('Train matrix:', X_train_processed.shape, 'Test matrix:', X_test_processed.shape)


The assignment reuses one name. Initially, `preprocessor` is the `ColumnTransformer` object. Calling `.fit(X_train)` fits that object and stores it as `preprocess`. Later, `preprocessor(data)` is a function that calls the fitted transformer's `.transform()`. The returned `preprocessed_data` is a numeric matrix. Grid search fits a fresh copy of the transformer in each training fold.

## save the baseline

In [ ]:
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

results, predictions, model_specs = {}, {}, {}
def evaluate(name, pred):
    predictions[name] = np.asarray(pred)
    results[name] = {'Accuracy': accuracy_score(y_test_labels, pred),
                    'Macro F1': f1_score(y_test_labels, pred, average='macro', zero_division=0)}
    print(name, {metric: round(value, 3) for metric, value in results[name].items()})

rf_baseline = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=1)
rf_baseline.fit(X_train_processed, y_train_labels)
print('Training accuracy:', rf_baseline.score(X_train_processed, y_train_labels))
Path('artifacts').mkdir(exist_ok=True)
joblib.dump({'preprocessor': preprocess, 'model': rf_baseline, 'classes': CLASS_NAMES},
            'artifacts/random_forest_baseline.joblib')

## evaluate the baseline

In [ ]:
evaluate('RF baseline', rf_baseline.predict(X_test_processed))
print(classification_report(y_test_labels, predictions['RF baseline'],
                            target_names=CLASS_NAMES, zero_division=0))

Macro F1 gives each happiness class equal weight. We compare these test scores with the training accuracy above to check how well the baseline generalizes.

In [ ]:
model_2 = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=2,
                                  random_state=SEED, n_jobs=1)
model_2.fit(X_train_processed, y_train_labels)
evaluate('RF adjusted', model_2.predict(X_test_processed))

We change 200 trees to 300, cap depth at 8 and raise the minimum leaf size from 1 to 2.  Test accuracy changes from 0.548 to 0.571, and macro F1 from 0.548 to 0.574. Correct predictions increase from 23 to 24 of the 42 countries.

We use four-fold stratified cross-validation to compare eight Random Forest settings by macro F1. This gives a systematic training-set comparison instead of repeatedly trying values against the test set. The pipeline fits its imputer, scaler and encoder within each fold.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {'classifier__n_estimators': np.arange(100, 301, 200),
              'classifier__max_depth': [None, 8],
              'classifier__min_samples_leaf': np.arange(1, 3)}
rf_pipeline = Pipeline([('preprocess', clone(preprocess)),
                        ('classifier', RandomForestClassifier(random_state=SEED, n_jobs=1))])
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
gridmodel = GridSearchCV(rf_pipeline, param_grid, cv=cv, scoring='f1_macro')
gridmodel.fit(X_train, y_train_labels)
print('Best CV macro F1:', round(gridmodel.best_score_, 3))
print('Best settings:', gridmodel.best_params_)
evaluate('RF grid search', gridmodel.predict(X_test))

In [ ]:
from sklearn.svm import SVC

svc_model = SVC(kernel='rbf', C=2, gamma='scale')
svc_model.fit(X_train_processed, y_train_labels)
evaluate('SVC', svc_model.predict(X_test_processed))
classical_models = {'RF baseline': rf_baseline, 'RF adjusted': model_2,
                    'RF grid search': gridmodel.best_estimator_.named_steps['classifier'],
                    'SVC': svc_model}
for name, estimator in classical_models.items():
    keys = (['n_estimators', 'max_depth', 'min_samples_leaf', 'max_features']
            if name.startswith('RF') else ['kernel', 'C', 'gamma'])
    model_specs[name] = {key: estimator.get_params()[key] for key in keys}

The RBF SVC allows a nonlinear boundary. `C=2` sets the penalty for classification errors, and `gamma='scale'` sets kernel width using feature variance and count.

We set aside 24 of the 95 training countries for validation. The neural-network preprocessor is fitted on the other 71 countries.

In [ ]:
import keras
from keras import layers
keras.utils.set_random_seed(SEED)
n_classes = len(CLASS_NAMES)
inner_fit_idx, inner_val_idx = train_test_split(
    np.arange(len(X_train)), test_size=.25,
    random_state=SEED, stratify=y_train_labels.to_numpy())
nn_order = np.r_[inner_fit_idx, inner_val_idx]
nn_split_at = len(inner_fit_idx)
nn_labels = y_train_labels.to_numpy()[nn_order]
nn_preprocess = clone(preprocess).fit(X_train.iloc[inner_fit_idx])
train_nn = nn_preprocess.transform(X_train.iloc[nn_order]).astype('float32')
test_nn = nn_preprocess.transform(X_test).astype('float32')
feature_count = train_nn.shape[1]
y_train_oh = pd.DataFrame(np.eye(n_classes, dtype='float32')[nn_labels], columns=CLASS_NAMES)
assert y_train_oh.shape == (len(X_train), 5)
assert nn_split_at == len(train_nn) - len(inner_val_idx)
print('Stratified neural fit/validation:', nn_split_at, '/', len(inner_val_idx))

keras_model = keras.Sequential([
    layers.Input(shape=(feature_count,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(n_classes, activation='softmax'),
])
keras_model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])
history = keras_model.fit(train_nn, y_train_oh.to_numpy(), batch_size=20,
                          epochs=300, validation_split=.25, verbose=0)
print('SGD 300 epochs | final train loss {:.3f}, validation loss {:.3f}'.format(
      history.history['loss'][-1], history.history['val_loss'][-1]))

With one-hot labels, categorical cross-entropy is the negative log probability assigned to the true class. SGD updates the weights using mini-batch gradients, with 20 countries per batch here. We keep it for the baseline, then compare Adam, which adapts the step size for each parameter. Validation loss determines when to stop.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ['loss', 'accuracy']):
    epochs = np.arange(1, len(history.history[metric]) + 1)
    ax.plot(epochs, history.history[metric], label='Training')
    ax.plot(epochs, history.history['val_' + metric], label='Validation')
    ax.set(xlabel='Epoch', ylabel=metric.title(), title='Baseline NN: ' + metric)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_base_epoch = int(np.argmin(history.history['val_loss'])) + 1
base_probs = keras_model.predict(test_nn, verbose=0)
prediction_column_index = base_probs.argmax(axis=1)
prediction_labels = [y_train_oh.columns[i] for i in prediction_column_index]
evaluate('NN SGD baseline', prediction_column_index)
model_specs['NN SGD baseline'] = {'Widths': '128-64-64-32-5', 'Activation': 'ReLU / softmax',
                                 'Optimizer': 'SGD (0.01)', 'Batch size': 20, 'Epochs': 300}

The lowest validation loss is 0.925 at epoch 121, compared with 1.290 at epoch 300. Final training loss is 0.147. Validation loss ends above its minimum despite the lower training loss, so we would not extend this run beyond 300 epochs.

We run four versions: neither method, Dropout, BatchNorm, and both. Each uses Adam (0.001), the same split and seed, and batches of 20.

Dropout is 0.20 after the first three hidden layers. BatchNorm is placed before each hidden activation.

In [ ]:
def make_network(activation='ReLU', dropout=False, batchnorm=False):
    network = keras.Sequential([layers.Input(shape=(feature_count,))])
    for i, width in enumerate((128, 64, 64, 32)):
        network.add(layers.Dense(width))
        if batchnorm:
            network.add(layers.BatchNormalization())
        network.add(layers.LeakyReLU(negative_slope=.1) if activation == 'LeakyReLU'
                    else layers.Activation(activation.lower()))
        if dropout and i < 3:
            network.add(layers.Dropout(.20))
    network.add(layers.Dense(n_classes, activation='softmax'))
    network.compile(optimizer=keras.optimizers.Adam(.001),
                    loss='categorical_crossentropy', metrics=['accuracy'])
    return network

networks, histories, validation_rows = {}, {}, {}
def fit_variant(name, activation='ReLU', dropout=False, batchnorm=False):
    keras.utils.set_random_seed(SEED)
    net = make_network(activation, dropout, batchnorm)
    stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    h = net.fit(train_nn, y_train_oh.to_numpy(), batch_size=20, epochs=100,
                validation_split=.25, callbacks=[stop], verbose=0).history
    best_epoch = int(np.argmin(h['val_loss'])) + 1
    train_metrics = net.evaluate(train_nn[:nn_split_at], y_train_oh.to_numpy()[:nn_split_at], verbose=0)
    val_metrics = net.evaluate(train_nn[nn_split_at:], y_train_oh.to_numpy()[nn_split_at:], verbose=0)
    networks[name], histories[name] = net, h
    validation_rows[name] = {'Train loss': train_metrics[0], 'Validation loss': val_metrics[0],
                             'Train accuracy': train_metrics[1], 'Validation accuracy': val_metrics[1],
                             'Best epoch': best_epoch, 'Epochs run': len(h['loss'])}
    model_specs[name] = {'Widths': '128-64-64-32-5', 'Activation': activation, 'Output': 'softmax',
                         'Optimizer': 'Adam (0.001)', 'Batch size': 20, 'Dropout': .20 if dropout else 0,
                         'BatchNorm': batchnorm, 'Best epoch': best_epoch, 'Patience': 15, 'Epoch cap': 100}

regularization_configs = {'NN Adam / ReLU': (False, False), 'NN Dropout': (True, False),
                          'NN BatchNorm': (False, True), 'NN Dropout + BatchNorm': (True, True)}
for name, (dropout, batchnorm) in regularization_configs.items():
    fit_variant(name, dropout=dropout, batchnorm=batchnorm)
reg_table = pd.DataFrame(validation_rows).T
chosen_variant = reg_table['Validation loss'].idxmin()
display(reg_table.round(3))
fig, axes = plt.subplots(2, 4, figsize=(17, 7), sharex='col', sharey='row')
for col, name in enumerate(regularization_configs):
    h = histories[name]
    for row, metric in enumerate(['loss', 'accuracy']):
        ax = axes[row, col]
        epochs = np.arange(1, len(h[metric]) + 1)
        ax.plot(epochs, h[metric], label='Training')
        ax.plot(epochs, h['val_' + metric], label='Validation')
        ax.set(title=name.replace('NN ', ''), xlabel='Epoch', ylabel=metric.title())
        ax.legend()
plt.tight_layout()
plt.show()

Using both methods gives validation loss 0.968, compared with 1.014 without regularization. Dropout alone gives 1.021, and BatchNorm alone gives 1.252.The table evaluates the restored weights in inference mode; the curves also include the training behavior of Dropout and BatchNorm.

In [ ]:
activation_names = {'ReLU': 'NN Adam / ReLU', 'LeakyReLU': 'NN LeakyReLU',
                    'Tanh': 'NN Tanh', 'Sigmoid': 'NN Sigmoid'}
for activation, name in activation_names.items():
    if activation != 'ReLU':
        fit_variant(name, activation=activation)
activation_table = pd.DataFrame({a: validation_rows[n] for a, n in activation_names.items()}).T
best_activation = activation_table['Validation loss'].idxmin()
display(activation_table.round(3))
for activation, name in activation_names.items():
    losses = histories[name]['val_loss']
    plt.plot(np.arange(1, len(losses) + 1), losses, label=activation)
plt.xlabel('Epoch')
plt.ylabel('Validation cross-entropy')
plt.title('Activation comparison: Adam, same split and stopping rule')
plt.legend()
plt.tight_layout()
plt.show()
for name, net in networks.items():
    evaluate(name, net.predict(test_nn, verbose=0).argmax(axis=1))

Sigmoid has the lowest validation loss (0.951). Sigmoid reaches its minimum at epoch 89; ReLU and tanh reach theirs at epochs 11 and 7. This supports choosing Sigmoid for this validation split, but does not identify the mechanism behind its advantage.

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='IProgress not found.*')
    import shap

tree_pipeline = gridmodel.best_estimator_
shap_test = tree_pipeline.named_steps['preprocess'].transform(X_test)
shap_feature_names = tree_pipeline.named_steps['preprocess'].get_feature_names_out()
shap_values = shap.TreeExplainer(tree_pipeline.named_steps['classifier'])(shap_test).values
assert shap_values.shape == (42, len(shap_feature_names), 5)
shap_rank = pd.Series(np.abs(shap_values).mean(axis=(0, 2)), index=shap_feature_names).sort_values(ascending=False)
shap.summary_plot([shap_values[:, :, k] for k in range(5)], shap_test,
                  feature_names=[s.replace('num__', '').replace('cat__', '') for s in shap_feature_names],
                  class_names=CLASS_NAMES, plot_type='bar', max_display=12, show=False, rng=np.random.default_rng(SEED))
plt.title('Grid-search Random Forest: SHAP by class')
plt.xlabel('Mean absolute SHAP contribution to class probability')
plt.tight_layout()
plt.show()

The leading tree features are social_support, gdp_x_social_support, gdp_per_capita. Each color represents a class.

In [ ]:
nn_model = networks[chosen_variant]
nn_shap_names = nn_preprocess.get_feature_names_out()
background = shap.sample(train_nn[:nn_split_at], 15, random_state=SEED)
foreground = test_nn[:12]
np.random.seed(SEED)
explainer = shap.KernelExplainer(
    lambda x: nn_model.predict(np.asarray(x, dtype='float32'), verbose=0), background)
nn_shap = np.asarray(explainer.shap_values(foreground, nsamples=300, l1_reg=0, silent=True))
assert nn_shap.shape == (12, len(nn_shap_names), 5)
nn_importance = pd.Series(np.abs(nn_shap).mean(axis=(0, 2)), index=nn_shap_names)
top_features = nn_importance.nlargest(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')
ax.barh(top_features.index.str.replace('num__', '', regex=False), top_features.values)
ax.set(xlabel='Mean absolute SHAP contribution (probability units)',
       title='Neural network SHAP: 12 test countries')
plt.show()

SHAP helps describe which inputs contribute most to its predictions. Here it uses 12 test countries, 15 training backgrounds and 300 sampled coalitions. correlated features and the background sample can change the attributions. It is easier to discuss the model's inputs with this plot, but the plot does not fully justify a prediction or identify causes of happiness.

The table includes all 12 fitted configurations. The ReLU/Adam run is reused for the activation comparison and counted once.

In [ ]:
comparison = pd.DataFrame(results).T.sort_values(['Macro F1', 'Accuracy'], ascending=False)
comparison['Correct / 42'] = [f"{int((predictions[name] == y_test_labels.to_numpy()).sum())}/42
                              for name in comparison.index]
assert len(comparison) == 12
assert set(comparison.index) == set(model_specs)
display(comparison[['Correct / 42', 'Accuracy', 'Macro F1']].round(3))
best_name = comparison.index[0]
display(pd.Series(model_specs[best_name], name=best_name).to_frame())

NN Dropout gives the highest test macro F1 (0.638) and accuracy 0.643 (27/42 countries). Its settings are listed above. With 19,525 parameters and only 71 fitting countries, restricting reliance on particular hidden units is a plausible reason Dropout helped. Our validation choices were NN Dropout + BatchNorm for regularization and Sigmoid for activation; the test winner was not the validation winner.

GDP contribution and social support track happiness class closely in the training data (Spearman 0.841 and 0.836). The tree's leading SHAP features are social_support, gdp_x_social_support, gdp_per_capita; the neural subset emphasizes social_support, freedom_to_make_life_choices, log_gni. We added log features for the skewed population and GNI columns and imputed missing values. 

The test set is small: one test country changes accuracy by 2.38 percentage points. The development indicators are also older than the happiness surveys, and the WHR factors already describe score contributions. Neural weights use only 71 countries, compared with 95 for classical fits. We would next try a smaller network across several training/validation splits and check whether the SHAP rankings change with different backgrounds.

### Course references
- **L3:** slides pp. 5, 12; *Neural Networks with Keras* recitation — activations, gradient updates, Sequential/Dense, Adam and SGD.
- **L4:** slides p. 3 — mini-batch training.
- **L5:** slides pp. 6–7 — softmax, one-hot labels and cross-entropy.
- **Project 1:** assignment instructions and the two supplied CSV files — architecture, preprocessing, classical models, regularization and SHAP.